# Handling Message Blocks

### Calling Claude with the JSON Schema
Helping it understand it has a tool available

In [1]:
from dotenv import load_dotenv
from utils.util_funcs import *

load_dotenv()
client, model = api_client_setup()

In [13]:
get_current_datetime_schema = ToolParam({
    "name": "get_current_datetime",
    "description": "Returns the current date and time formatted according to the specified format",
    "input_schema": {
        "type": "object",
        "properties": {
            "date_format": {
                "type": "string",
                "description": "A string specifying the format of the returned datetime. Uses Python's strftime format codes.",
                "default": "%Y-%m-%d %H:%M:%S"
            }
        },
        "required": []
    }
})

In [14]:
messages = []

messages.append({ 
    "role": "user",
    "content": "Can you give me the exact time, given to me in hours, minutes and seconds?"
})
messages

[{'role': 'user',
  'content': 'Can you give me the exact time, given to me in hours, minutes and seconds?'}]

In [15]:
response = client.messages.create(
    model = model,
    max_tokens = 1000, 
    messages = messages, 
    tools = [get_current_datetime_schema]
)

messages.append({ 
    "role": "assistant",
    "content": response.content
})
messages

[{'role': 'user',
  'content': 'Can you give me the exact time, given to me in hours, minutes and seconds?'},
 {'role': 'assistant',
  'content': [ToolUseBlock(id='toolu_01GK5ikBZGQLy6bKubwr2KgD', caller=DirectCaller(type='direct'), input={'date_format': '%H:%M:%S'}, name='get_current_datetime', type='tool_use')]}]

There is now a new object called a `ToolUseBlock`, in addition to the `TextBlock` object

In [5]:
result = get_current_datetime(**response.content[0].input)
result

'14:58:04'

In [16]:
messages.append({ 
    "role": "user", 
    "content": [
        { 
            "type": "tool_result",
            "tool_use_id": response.content[0].id,
            "content": result,
            "is_error": False
        }
    ]
})

messages

[{'role': 'user',
  'content': 'Can you give me the exact time, given to me in hours, minutes and seconds?'},
 {'role': 'assistant',
  'content': [ToolUseBlock(id='toolu_01GK5ikBZGQLy6bKubwr2KgD', caller=DirectCaller(type='direct'), input={'date_format': '%H:%M:%S'}, name='get_current_datetime', type='tool_use')]},
 {'role': 'user',
  'content': [{'type': 'tool_result',
    'tool_use_id': 'toolu_01GK5ikBZGQLy6bKubwr2KgD',
    'content': '14:58:04',
    'is_error': False}]}]

In [17]:
messages

[{'role': 'user',
  'content': 'Can you give me the exact time, given to me in hours, minutes and seconds?'},
 {'role': 'assistant',
  'content': [ToolUseBlock(id='toolu_01GK5ikBZGQLy6bKubwr2KgD', caller=DirectCaller(type='direct'), input={'date_format': '%H:%M:%S'}, name='get_current_datetime', type='tool_use')]},
 {'role': 'user',
  'content': [{'type': 'tool_result',
    'tool_use_id': 'toolu_01GK5ikBZGQLy6bKubwr2KgD',
    'content': '14:58:04',
    'is_error': False}]}]

In [18]:
client.messages.create(
    model = model,
    max_tokens = 1000,
    messages = messages,
    tools = [get_current_datetime_schema]
)

Message(id='msg_011Cd73gXP8aqN4QqFTAnnah', container=None, content=[TextBlock(citations=None, text='The exact time right now is **14:58:04** (2:58:04 PM in 12-hour format).', type='text')], model='claude-haiku-4-5-20251001', role='assistant', stop_details=None, stop_reason='end_turn', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=702, output_tokens=31, output_tokens_details=None, server_tool_use=None, service_tier='standard'))